# 01 · The tensile tests as data: two ontology generations, one neutral record

**Goal.** Show how the same tensile test is described in two generations of the Platform MaterialDigital core ontology, PMDco 2 and PMDco 3, both used together with the Tensile Test Ontology (TTO), and how an *ontology profile* maps both descriptions to the same neutral records that the analysis needs.

**Audience.** Ontologists and RDF experts. MSE domain experts read it after the method, to see where the analysed numbers come from.

**Prerequisites.**
- RDF and SPARQL basics.
- The `pmd-demonstrator` environment from `environment.yml`, which installs the package `pmd_tensile` from this repository (editable).
- `ONTODOCKER_ADDRESS` (and `ONTODOCKER_TOKEN`, if the instance needs one) set in the environment before Jupyter starts. Neither is printed here.

**Reading paths.** MSE domain experts: [05](05-tensile-analysis-overview.ipynb) → [02](02-tensile-method.ipynb) → **01** → [06](06-tensile-results-publication.ipynb). Ontologists and RDF experts: **01** → [04](04-tensile-workflow-knowledge-graph.ipynb) → [05](05-tensile-analysis-overview.ipynb) → [06](06-tensile-results-publication.ipynb). Workflow developers: [03](03-tensile-workflow-building.ipynb) → [04](04-tensile-workflow-knowledge-graph.ipynb) → [02](02-tensile-method.ipynb) → [05](05-tensile-analysis-overview.ipynb) → [06](06-tensile-results-publication.ipynb). Readers new to Ontodocker and the Dataportal may start with [00](00-infrastructure-tour.ipynb).

The ten tests on S355 steel (Zx1–Zx4 in rolling direction, Zy1–Zy4 perpendicular to it, Zd2 and Zd3 diagonal) are described in both generations. Zx3 is the example throughout.

| Symbol | Meaning | How it is obtained here | Unit |
| --- | --- | --- | --- |
| S0, s1–s3 | original cross-section, and its individual measurements | read from the graph | mm² |
| L0, Lu | original and final gauge length | read from the graph | mm |
| Su | cross-section after fracture | read from the graph | mm² |
| Fm, Rm, ReH, Rp0.2, E, A, Z | the lab's reported properties (maximum force, tensile strength, upper yield strength, proof strength, modulus of elasticity as reported by the lab, elongation after fracture, reduction of area) | read from the graph | N, MPa, GPa, % |

How these properties are computed from the curves is the subject of [notebook 02](02-tensile-method.ipynb).

| Section | Question |
| --- | --- |
| 1. Six datasets on Ontodocker | Which graphs are hosted, and how do the hosted PMDco 3 graphs relate to the committed files? |
| 2. Zx3 in both generations | How are S0, ReH and a curve column expressed? |
| 3. Ontology profiles | Where is the ontology-specific knowledge kept, and how is a graph checked against it? |
| 4. The same neutral records | Do both descriptions give the analysis the same input? |
| 5. The curve tables | How does each graph say where the raw curve is and what its columns mean? |

*Execution class: network-read.* SPARQL queries and one Turtle download from Ontodocker, two curve downloads and one header read from the public data host. One committed file in `datasets/` is read for comparison. Nothing is written to remote services, the repository or the disk.

## Setup

The Ontodocker datasets are read with [praeco](https://github.com/materialdigital/praeco), either directly (praeco's `OntodockerClient`, built by `sources.ontodocker_client()` from the environment) or through `sources.select`, which runs a SPARQL SELECT on every dataset of a source. `short()` writes IRIs with the prefixes below; both generations use the same namespace `https://w3id.org/pmd/co/` (`pmdco:`), PMDco 2 with readable local names and PMDco 3 with numeric IDs.

In [1]:
import dataclasses
import os
import re
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import rdflib
from IPython.display import display

import pmd_tensile
from pmd_tensile import profiles, sources
from pmd_tensile.nodes.acquisition import fetch_curve, get_reported_properties, get_specimen_record
from pmd_tensile.nodes.exploration import check_conformance, list_specimens
from pmd_tensile.profiles import get_profile, pmdco2_tto, pmdco3_tto

pd.set_option("display.max_colwidth", None)  # pandas would otherwise cut IRIs, titles and evidence.

if not os.environ.get(sources.ONTODOCKER_ADDRESS_ENV, "").strip():
    raise RuntimeError("Set ONTODOCKER_ADDRESS (and ONTODOCKER_TOKEN if required) before starting Jupyter.")

ORIENTATIONS = ("parallel", "perpendicular", "diagonal")
GENERATIONS = {  # profile ID -> the three hosted datasets of that generation
    "pmdco2-tto": [f"pmdco2_tto_example_{orientation}" for orientation in ORIENTATIONS],
    "pmdco3-tto": [f"pmdco3_tto_example_{orientation}" for orientation in ORIENTATIONS],
}
SOURCES = {profile_id: {"kind": "ontodocker", "datasets": datasets} for profile_id, datasets in GENERATIONS.items()}
SPECIMEN = "Zx3"

# Committed files in datasets/, found through the editable install of pmd_tensile from this repository.
DATASETS_DIR = Path(pmd_tensile.__file__).resolve().parents[1] / "datasets"
if not (DATASETS_DIR / "pmdco3_tto_example.ttl").is_file():
    raise RuntimeError("datasets/ not found: install pmd_tensile editable from this repository (environment.yml).")

PREFIXES = {
    "pmdco": "https://w3id.org/pmd/co/",
    "tto": "https://w3id.org/pmd/tto/",
    "obo": "http://purl.obolibrary.org/obo/",
    "tte": "https://w3id.org/pmd/ao/tte/",  # PMDco 2 data
    "abox": "https://w3id.org/pmd/demodata/tensiletest_S355/",  # PMDco 3 data
    "csvw": "http://www.w3.org/ns/csvw#",
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
    "owl": "http://www.w3.org/2002/07/owl#",
    "xsd": "http://www.w3.org/2001/XMLSchema#",
    "dc": "http://purl.org/dc/elements/1.1/",
    "dcterms": "http://purl.org/dc/terms/",
    "unit": "http://qudt.org/vocab/unit/",
    "orcid": "https://orcid.org/",
}
SPARQL_PREFIXES = "\n".join(f"PREFIX {prefix}: <{namespace}>" for prefix, namespace in PREFIXES.items())
# Labels of the PMDco 3/TTO/OBO terms from the ontology files at TTO commit 08a7aec, kept by the profile.
LABELS = {curie: label for curie, label, _ in pmdco3_tto.IRI_LABELS}


def short(text) -> str:
    """An IRI as prefix:name; anything else unchanged."""
    text = "" if text is None else str(text)
    for prefix, namespace in PREFIXES.items():
        if text.startswith(namespace):
            return f"{prefix}:{text[len(namespace):]}"
    return text


def short_n3(text: str) -> str:
    """Shorten every <IRI> in an N3 term or triple."""
    return re.sub(r"<([^>]+)>", lambda match: short(match.group(1)), text)


def select(dataset: str, query: str, columns: list[str]) -> pd.DataFrame:
    """SPARQL SELECT on one hosted dataset (the package's Ontodocker access)."""
    rows = sources.select({"kind": "ontodocker", "datasets": [dataset]}, SPARQL_PREFIXES + "\n" + query, columns)
    return rows.drop(columns="source_part")


print("Profiles:", ", ".join(GENERATIONS))
print("Example specimen:", SPECIMEN)

Profiles: pmdco2-tto, pmdco3-tto
Example specimen: Zx3


## 1. Six datasets on Ontodocker

Ontodocker hosts each orientation group of the tests as one dataset, in both generations. The PMDco 2 datasets were already hosted; the PMDco 3 ones were cut by this demonstrator from the original PMDco 3 graph of the TTO repository (commit `08a7aec`) into the same three groups, with per-test curve tables added (section 5), and each title extended by its orientation (`scripts/slice_pmdco3.py`).

The table lists the six datasets of the two generations, whether Ontodocker hosts each, and the number of tensile tests in each (class `pmdco:TensileTest` in PMDco 2, `pmdco:PMD_0000974`, "tensile testing process", in PMDco 3) and the title of the ontology header.

In [2]:
retrieved = pd.Timestamp.now(tz="Europe/Berlin").floor("s")
# praeco's OntodockerClient, built by the package from the environment and not kept in a variable
available = set(sources.ontodocker_client().datasets.list())

TESTS_AND_TITLE = """
SELECT ?tests ?title WHERE {
  { SELECT (COUNT(DISTINCT ?test) AS ?tests) WHERE { VALUES ?class { pmdco:TensileTest pmdco:PMD_0000974 } ?test a ?class } }
  OPTIONAL { ?ontology a owl:Ontology ; dc:title ?title }
}
"""
overview = []
for profile_id, names in GENERATIONS.items():
    for name in names:
        row = {"dataset": name, "generation": "PMDco 2" if profile_id == "pmdco2-tto" else "PMDco 3",
               "hosted": name in available, "tensile tests": None, "dc:title": "–"}
        if row["hosted"]:
            answer = select(name, TESTS_AND_TITLE, ["tests", "title"]).iloc[0]
            row.update({"tensile tests": int(answer["tests"]), "dc:title": answer["title"]})
        overview.append(row)
display(pd.DataFrame(overview).set_index("dataset"))
print("Data retrieved:", retrieved)

,generation,hosted,tensile tests,dc:title
dataset,,,,
pmdco2_tto_example_parallel,PMDco 2,True,4,Tensile Test Ontology (TTO) data mapping example
pmdco2_tto_example_perpendicular,PMDco 2,True,4,Tensile Test Ontology (TTO) data mapping example
pmdco2_tto_example_diagonal,PMDco 2,True,2,Tensile Test Ontology (TTO) data mapping example
pmdco3_tto_example_parallel,PMDco 3,True,4,TTO/PMDco 3.0 A-Box mapping example of S355 steel tensile test data (in rolling direction)
pmdco3_tto_example_perpendicular,PMDco 3,True,4,TTO/PMDco 3.0 A-Box mapping example of S355 steel tensile test data (perpendicular to rolling direction)
pmdco3_tto_example_diagonal,PMDco 3,True,2,TTO/PMDco 3.0 A-Box mapping example of S355 steel tensile test data (diagonal to rolling direction)


Data retrieved: 2026-11-10 20:39:09+01:00


### Hosted and committed PMDco 3 graphs

The slices are committed in `datasets/` and were uploaded unchanged. Ontodocker returns them with one systematic difference: literals written as `"…"^^xsd:string` come back as plain literals `"…"`. Under RDF 1.1 these are the same term, but rdflib 7 keeps them apart, so a strict comparison in rdflib reports a difference. The comparison below therefore maps `xsd:string` to plain literals first. Blank nodes (the RDF lists of the curve-table columns) are only counted, because a full isomorphism check is slow on graphs with many blank nodes.

In [3]:
XSD_STRING = rdflib.XSD.string


def plain_strings(graph: rdflib.Graph) -> set:
    """Triples without blank nodes, with xsd:string literals written as plain literals."""
    mapped = set()
    for s, p, o in graph:
        if isinstance(s, rdflib.BNode) or isinstance(o, rdflib.BNode):
            continue
        if isinstance(o, rdflib.Literal) and o.datatype == XSD_STRING:
            o = rdflib.Literal(str(o))
        mapped.add((s, p, o))
    return mapped


def count(graph: rdflib.Graph, condition) -> int:
    return sum(1 for triple in graph if condition(triple))


pmdco3_parallel_text = sources.ontodocker_client().datasets.fetch_turtle("pmdco3_tto_example_parallel")
graphs = {
    "hosted (Ontodocker)": rdflib.Graph().parse(data=pmdco3_parallel_text, format="turtle"),
    "committed (datasets/pmdco3_tto_example_parallel.ttl)": rdflib.Graph().parse(
        DATASETS_DIR / "pmdco3_tto_example_parallel.ttl", format="turtle"
    ),
}
display(pd.DataFrame({
    name: {
        "triples": len(graph),
        "literals typed xsd:string": count(graph, lambda t: isinstance(t[2], rdflib.Literal) and t[2].datatype == XSD_STRING),
        "triples with a blank node": count(graph, lambda t: isinstance(t[0], rdflib.BNode) or isinstance(t[2], rdflib.BNode)),
    }
    for name, graph in graphs.items()
}))
hosted_graph, committed_graph = graphs.values()
print("Triples without blank nodes equal after mapping xsd:string to plain literals:",
      plain_strings(hosted_graph) == plain_strings(committed_graph))

,hosted (Ontodocker),committed (datasets/pmdco3_tto_example_parallel.ttl)
triples,3151,3151
literals typed xsd:string,0,360
triples with a blank node,36,36


Triples without blank nodes equal after mapping xsd:string to plain literals: True


## 2. Zx3 in both generations

Three things the analysis needs, for one specimen: the original cross-section S0, the lab-reported upper yield strength ReH, and the force column of the raw curve. One small query per generation finds them for the test piece named `Zx3`, in the dataset of its orientation group.

In [4]:
ZX3_PMDCO2 = """
SELECT ?quantity ?node ?class ?value ?unit (DATATYPE(?value) AS ?datatype) WHERE {
  ?test a pmdco:TensileTest ;
        pmdco:input/pmdco:characteristic ?name .
  ?name a pmdco:TestPieceName ; pmdco:value "Zx3" .
  {
    ?test pmdco:characteristic ?node .
    ?node a pmdco:CrossSectionArea ; pmdco:value ?value ; pmdco:unit ?unit .
    FILTER(STRENDS(STR(?node), "_S0"))  # S0, Su and s1..s3 share the class; only the IRI tells them apart
    BIND("S0" AS ?quantity) BIND(pmdco:CrossSectionArea AS ?class)
  } UNION {
    ?test pmdco:characteristic ?node .
    ?node a pmdco:UpperYieldStrength ; pmdco:value ?value ; pmdco:unit ?unit .
    BIND("ReH" AS ?quantity) BIND(pmdco:UpperYieldStrength AS ?class)
  } UNION {
    ?test pmdco:characteristic/pmdco:resource/csvw:tableSchema/csvw:column/rdf:rest*/rdf:first ?node .
    ?node a pmdco:Force ; pmdco:unit ?unit .
    BIND("force column" AS ?quantity) BIND(pmdco:Force AS ?class)
  }
}
"""

ZX3_PMDCO3 = """
SELECT ?quantity ?node ?class ?value ?unit (DATATYPE(?value) AS ?datatype) WHERE {
  ?test a pmdco:PMD_0000974 ;                                # tensile testing process
        obo:OBI_0000293 ?test_piece .                        # has specified input
  ?identifier obo:IAO_0000219 ?test_piece ;                  # denotes
              obo:OBI_0001938/pmdco:PMD_0000006 "Zx3" .      # has value specification / has value
  {
    ?test_piece obo:RO_0000086 ?node .                       # has quality
    ?node a tto:TTO_0000026 .
    BIND("S0" AS ?quantity) BIND(tto:TTO_0000026 AS ?class)
  } UNION {
    ?test_piece pmdco:PMD_0020005/obo:RO_0000086 ?node .     # test piece -> material ("porosity" in the ontology files), has quality
    ?node a tto:TTO_0000059 .
    BIND("ReH" AS ?quantity) BIND(tto:TTO_0000059 AS ?class)
  } UNION {
    ?test obo:OBI_0000299 ?table .                           # has specified output
    ?table a pmdco:PMD_0000992 ;                             # time series
           csvw:tableSchema/csvw:column/rdf:rest*/rdf:first ?node .
    ?node obo:IAO_0000136 ?entity ;                          # is about
          obo:IAO_0000039 ?unit .                            # has measurement unit label
    ?entity a pmdco:PMD_0020200 .
    BIND("force column" AS ?quantity) BIND(pmdco:PMD_0020200 AS ?class)
  }
  OPTIONAL {
    ?test obo:OBI_0000299 ?datum .                           # has specified output
    ?datum a obo:IAO_0000109 ;                               # measurement datum
           obo:IAO_0000136 ?node ;                           # is about
           obo:OBI_0001938 ?spec .                           # has value specification
    ?spec pmdco:PMD_0000006 ?value ;                         # has value
          obo:IAO_0000039 ?unit .                            # has measurement unit label
  }
}
"""

COLUMNS = ["quantity", "node", "class", "value", "unit", "datatype"]
side_by_side = []
for profile_id, query, dataset in (
    ("pmdco2-tto", ZX3_PMDCO2, "pmdco2_tto_example_parallel"),
    ("pmdco3-tto", ZX3_PMDCO3, "pmdco3_tto_example_parallel"),
):
    profile = get_profile(profile_id)
    for row in select(dataset, query, COLUMNS).to_dict("records"):
        side_by_side.append({
            "quantity": row["quantity"],
            "generation": "PMDco 2" if profile_id == "pmdco2-tto" else "PMDco 3",
            "node": short(row["node"]) if str(row["node"]).startswith("http") else "(blank node)",
            "class": short(row["class"]),
            "label of the class": LABELS.get(short(row["class"]), "–"),
            "value": row["value"] if row["value"] is not None else "–",
            "datatype": short(row["datatype"]) if row["datatype"] else "–",
            "unit": short(row["unit"]),
            "unit for the analysis": profile.unit_symbol(row["unit"]),
        })
order = {"S0": 0, "ReH": 1, "force column": 2}
side_by_side = pd.DataFrame(side_by_side)
display(side_by_side.assign(order=side_by_side["quantity"].map(order))
        .sort_values(["order", "generation"]).drop(columns="order").set_index(["quantity", "generation"]))

node  \
quantity     generation                                                            
S0           PMDco 2                 tte:pmdao-tto-tt-S355-3_crossSectionArea_S0   
             PMDco 3     abox:pmdao-tto-tt-S355-3_original_cross_section_area_S0   
ReH          PMDco 2                  tte:pmdao-tto-tt-S355-3_upperYieldStrength   
             PMDco 3           abox:pmdao-tto-tt-S355-3_upper_yield_strength_ReH   
force column PMDco 2                                                (blank node)   
             PMDco 3                                abox:Zx3_csvw_table/col/Load   

                                            class  \
quantity     generation                             
S0           PMDco 2       pmdco:CrossSectionArea   
             PMDco 3              tto:TTO_0000026   
ReH          PMDco 2     pmdco:UpperYieldStrength   
             PMDco 3              tto:TTO_0000059   
force column PMDco 2                  pmdco:Force   
             PMDco 3            pmdco:PMD_0020200   

                                                           label of the class  \
quantity     generation                                                         
S0           PMDco 2                                                        –   
             PMDco 3     original cross-sectional area of the parallel length   
ReH          PMDco 2                                                        –   
             PMDco 3                                     upper yield strength   
force column PMDco 2                                                        –   
             PMDco 3                                                    force   

                           value    datatype          unit  \
quantity     generation                                      
S0           PMDco 2     120.396  xsd:string  unit:MilliM2   
             PMDco 3     120.396   xsd:float  unit:MilliM2   
ReH          PMDco 2       379.0  xsd:string   unit:MegaPa   
             PMDco 3       379.0   xsd:float   unit:MegaPa   
force column PMDco 2           –           –    unit:KiloN   
             PMDco 3           –           –    unit:KiloN   

                        unit for the analysis  
quantity     generation                        
S0           PMDco 2                      mm2  
             PMDco 3                      mm2  
ReH          PMDco 2                      MPa  
             PMDco 3                      MPa  
force column PMDco 2                       kN  
             PMDco 3                       kN

The two generations say the same about Zx3, but in different shapes:

- **PMDco 2** hangs each value directly on a node that is a `pmdco:characteristic` of the test, with `pmdco:value` and `pmdco:unit`; the numbers are plain string literals. S0, Su and s1–s3 share the class `pmdco:CrossSectionArea`, so only the local name of the IRI (`…_crossSectionArea_S0`) says which one is S0. The force column of the curve table is a blank node typed directly with the quantity class `pmdco:Force`.
- **PMDco 3** separates the *quality* (S0 is a quality of the test piece, ReH a quality of its material; the data reach the material through `pmdco:PMD_0020005`, which the pinned ontology files label *porosity*, an `owl:Class`) from the *measurement datum* that is about it. The number, typed `xsd:float`, and the unit sit on the datum's value specification. S0 has its own class. Classes are numeric IDs; their labels are not in the data graphs and come from the ontology files (here from the profile's table `pmdco3_tto.IRI_LABELS`, taken from TTO at commit `08a7aec`). In the curve tables that this demonstrator added (section 5), the force column is itself a measurement datum that is about a force.
- Both use the unit IRI `unit:MegaPa` for ReH, which is not a QUDT IRI (QUDT writes `unit:MegaPA`). The profiles map it to MPa by a declared alias; any other unknown unit raises an error.

## 3. Ontology profiles

The analysis in notebooks 02–05 never sees an IRI. Everything ontology-specific is in an *ontology profile*, a declarative object in `pmd_tensile.profiles` that holds SPARQL text and conventions, but no graph-access code:

- **conformance ASKs**, checked before anything is read;
- SELECT queries with fixed result columns for the **specimens** (test, test-piece name, orientation), the **quantities** (S0, s1–s3, L0, Lu, Su, the test-speed switch point, and the lab-reported values), the **notes**, and the **curve columns** (curve URL, column position, role and unit);
- **unit aliases** for non-QUDT unit IRIs found in the data.

Supporting another ontology means writing another profile; the exploration and acquisition code is shared.

In [5]:
display(pd.DataFrame([
    {
        "profile": profile.profile_id,
        "description": profile.description,
        "conformance ASKs": len(profile.conformance_asks),
        "SELECT queries": ", ".join(
            name for name in ("specimens", "quantities", "notes", "curve_columns") if getattr(profile, name) is not None
        ),
        "unit aliases": "; ".join(f"{short(alias)} → {short(canonical)}" for alias, canonical in profile.unit_aliases.items()),
    }
    for profile in map(get_profile, GENERATIONS)
]).set_index("profile"))

# One conformance ASK as the profile holds it (with the profile's prefixes).
name, ask = next(iter(pmdco2_tto.CONFORMANCE_ASKS.items()))
print(f"pmdco2-tto, ASK '{name}':")
print(ask.strip())

,description,conformance ASKs,SELECT queries,unit aliases
profile,,,,
pmdco2-tto,PMDco 2.x + TTO (v2-era) S355 mapping; curves linked via CSVW; S0/Su/s1..sn told apart by IRI naming convention.,4,"specimens, quantities, notes, curve_columns",unit:kiloN → unit:KiloN; unit:MegaPa → unit:MegaPA; unit:GigaPa → unit:GigaPA
pmdco3-tto,"PMDco 3.x + TTO S355 mapping; specimens, geometry and reported values from the graph; curves from the per-test CSVW curve tables of the demonstrator's slices. The original graph has none and needs the explicit, NON-SEMANTIC CURVE_LOCATOR.",3,"specimens, quantities, notes, curve_columns",unit:MegaPa → unit:MegaPA; unit:GigaPa → unit:GigaPA


pmdco2-tto, ASK 'tensile test with test-piece name':
PREFIX pmd:  <https://w3id.org/pmd/co/>
PREFIX csvw: <http://www.w3.org/ns/csvw#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

ASK {
  ?process a pmd:TensileTest ;
           pmd:input ?test_piece .
  ?test_piece a pmd:TestPiece ;
              pmd:characteristic ?name .
  ?name a pmd:TestPieceName ;
        pmd:value ?specimen_id .
}


### Conformance checks

Each profile's ASK queries must all hold in every dataset of a source before the workflow lists any specimen. Run on its own generation, every ASK holds. Run on the other generation, none does, and the workflow's first node, `check_conformance`, stops with an error.

In [6]:
for profile_id, source in SOURCES.items():
    print(f"{profile_id} on its own datasets:")
    display(pd.DataFrame(profiles.conformance_results(profile_id, source)).T)

for profile_id, other in (("pmdco2-tto", "pmdco3_tto_example_parallel"), ("pmdco3-tto", "pmdco2_tto_example_parallel")):
    answers = profiles.conformance_results(profile_id, {"kind": "ontodocker", "datasets": [other]})
    held = sum(answer for by_dataset in answers.values() for answer in by_dataset.values())
    print(f"{profile_id} on {other}: {held} of {len(answers)} ASKs hold")

try:
    check_conformance("pmdco3-tto", {"kind": "ontodocker", "datasets": ["pmdco2_tto_example_parallel"]})
except profiles.ConformanceError as error:
    print(f"check_conformance: {type(error).__name__}: {error}")

pmdco2-tto on its own datasets:


,pmdco2_tto_example_parallel,pmdco2_tto_example_perpendicular,pmdco2_tto_example_diagonal
tensile test with test-piece name,True,True,True
cross-section area with value and unit,True,True,True
orientation (MaterialRelated),True,True,True
CSVW curve table with column schema,True,True,True


pmdco3-tto on its own datasets:


,pmdco3_tto_example_parallel,pmdco3_tto_example_perpendicular,pmdco3_tto_example_diagonal
tensile testing process with test-piece identifier,True,True,True
"original cross-section S0 of the test piece, measured with unit",True,True,True
orientation quality of the test piece's material,True,True,True


pmdco2-tto on pmdco3_tto_example_parallel: 0 of 4 ASKs hold


pmdco3-tto on pmdco2_tto_example_parallel: 0 of 3 ASKs hold


check_conformance: ConformanceError: source does not conform to profile 'pmdco3-tto'; failed ASKs: {'tensile testing process with test-piece identifier': ['pmdco2_tto_example_parallel'], 'original cross-section S0 of the test piece, measured with unit': ['pmdco2_tto_example_parallel'], "orientation quality of the test piece's material": ['pmdco2_tto_example_parallel']}


## 4. The same neutral records

The workflow's first steps are the exploration nodes (`check_conformance`, `list_specimens`) and the acquisition nodes (`get_specimen_record`, `get_reported_properties`, `fetch_curve`). They are the only nodes that know an ontology, and they take the profile ID and the source as plain inputs. A node is a plain Python function, so here they are called directly, once per generation.

Their outputs are the neutral records of `pmd_tensile.contract`, in fixed units (N, mm, mm², strain as a ratio, MPa, GPa, %). Every unit is converted through an explicit table.

In [7]:
listed, tables = {}, {}
for profile_id, source in SOURCES.items():
    checked = check_conformance(profile_id, source)  # raises if an ASK fails; returns the profile ID
    listed[profile_id] = list_specimens(checked, source)
    tables[profile_id] = profiles.specimen_table(profile_id, source).set_index("specimen_id")

pmdco2_table, pmdco3_table = tables["pmdco2-tto"], tables["pmdco3-tto"]
display(pd.DataFrame({
    "orientation (PMDco 2)": pmdco2_table["orientation"],
    "orientation (PMDco 3)": pmdco3_table["orientation"],
    "test (PMDco 2)": pmdco2_table["process"].map(short),
    "test (PMDco 3)": pmdco3_table["process"].map(short),
}))
print("Same specimen IDs:", listed["pmdco2-tto"] == listed["pmdco3-tto"])
print("Same orientation for every specimen:", pmdco2_table["orientation"].equals(pmdco3_table["orientation"]))

,orientation (PMDco 2),orientation (PMDco 3),test (PMDco 2),test (PMDco 3)
specimen_id,,,,
Zd2,diagonal to rolling direction,diagonal to rolling direction,tte:pmdao-tto-tt-S355-9_process,abox:pmdao-tto-tt-S355-9_process
Zd3,diagonal to rolling direction,diagonal to rolling direction,tte:pmdao-tto-tt-S355-10_process,abox:pmdao-tto-tt-S355-10_process
Zx1,in rolling direction,in rolling direction,tte:pmdao-tto-tt-S355-1_process,abox:pmdao-tto-tt-S355-1_process
Zx2,in rolling direction,in rolling direction,tte:pmdao-tto-tt-S355-2_process,abox:pmdao-tto-tt-S355-2_process
Zx3,in rolling direction,in rolling direction,tte:pmdao-tto-tt-S355-3_process,abox:pmdao-tto-tt-S355-3_process
Zx4,in rolling direction,in rolling direction,tte:pmdao-tto-tt-S355-4_process,abox:pmdao-tto-tt-S355-4_process
Zy1,perpendicular to rolling direction,perpendicular to rolling direction,tte:pmdao-tto-tt-S355-5_process,abox:pmdao-tto-tt-S355-5_process
Zy2,perpendicular to rolling direction,perpendicular to rolling direction,tte:pmdao-tto-tt-S355-6_process,abox:pmdao-tto-tt-S355-6_process
Zy3,perpendicular to rolling direction,perpendicular to rolling direction,tte:pmdao-tto-tt-S355-7_process,abox:pmdao-tto-tt-S355-7_process


Same specimen IDs: True
Same orientation for every specimen: True


For Zx3, the specimen record (geometry, orientation, lab notes, curve location and column semantics), the lab-reported values and the raw curve, from each generation:

In [8]:
records, reported, curves = {}, {}, {}
for profile_id, source in SOURCES.items():
    records[profile_id] = get_specimen_record(profile_id, source, SPECIMEN)
    reported[profile_id] = get_reported_properties(profile_id, source, SPECIMEN)
    curves[profile_id] = fetch_curve(records[profile_id])


def columns_text(columns) -> str:
    return "; ".join(f"{role}: column {spec.index + 1}, {spec.unit}" for role, spec in columns.items())


def side_by_side_fields(objects: dict) -> pd.DataFrame:
    rows = []
    for field in dataclasses.fields(objects["pmdco2-tto"]):
        values = [getattr(objects[profile_id], field.name) for profile_id in GENERATIONS]
        shown = [columns_text(value) if field.name == "curve_columns" else value for value in values]
        rows.append({"field": field.name, "PMDco 2": shown[0], "PMDco 3": shown[1], "equal": values[0] == values[1]})
    return pd.DataFrame(rows).set_index("field")


record_table = side_by_side_fields(records)
reported_table = side_by_side_fields(reported)
display(record_table)
display(reported_table)

pmdco2_curve, pmdco3_curve = curves["pmdco2-tto"], curves["pmdco3-tto"]
display(pd.DataFrame({
    channel: {
        "samples": len(getattr(pmdco2_curve, channel)),
        "identical arrays": np.array_equal(getattr(pmdco2_curve, channel), getattr(pmdco3_curve, channel), equal_nan=True),
    }
    for channel in ("time_s", "crosshead_mm", "force_n", "strain")
}).T)

differing = list(record_table.index[~record_table["equal"]]) + list(reported_table.index[~reported_table["equal"]])
print(f"Fields compared: {len(record_table)} of the specimen record, {len(reported_table)} of the lab-reported values.")
print("Fields that differ:", ", ".join(differing) or "none")
print("Same curve bytes (SHA-256):", pmdco2_curve.sha256 == pmdco3_curve.sha256, pmdco2_curve.sha256)

,PMDco 2,PMDco 3,equal
field,,,
specimen_id,Zx3,Zx3,True
provenance_iri,https://w3id.org/pmd/ao/tte/pmdao-tto-tt-S355-3_process,https://w3id.org/pmd/demodata/tensiletest_S355/pmdao-tto-tt-S355-3_process,False
orientation,in rolling direction,in rolling direction,True
s0_mm2,120.396,120.396,True
cross_section_measurements_mm2,"(120.323, 120.431, 120.433)","(120.323, 120.431, 120.433)",True
l0_mm,60.0,60.0,True
lu_mm,81.1,81.1,True
su_mm2,50.502,50.502,True
notes,"(In Walzrichtung aus Stahlblech entnommen; Extensometer nach 3,9 % Dehnung der Proben entfernt. Keine Erhöhung der Geschwindigkeit.,)","(In Walzrichtung aus Stahlblech entnommen; Extensometer nach 3,9 % Dehnung der Proben entfernt. Keine Erhöhung der Geschwindigkeit.,)",True


,PMDco 2,PMDco 3,equal
field,,,
specimen_id,Zx3,Zx3,True
e_gpa,199.0,199.0,True
rp02_mpa,376.0,376.0,True
reh_mpa,379.0,379.0,True
rel_mpa,None,None,True
rm_mpa,504.0,504.0,True
fm_n,60680.0,60680.0,True
a_percent,35.0,35.0,True
z_percent,58.0,58.0,True


,samples,identical arrays
time_s,13134,True
crosshead_mm,13134,True
force_n,13134,True
strain,13134,True


Fields compared: 12 of the specimen record, 9 of the lab-reported values.
Fields that differ: provenance_iri
Same curve bytes (SHA-256): True d0b3aaa7d4e871ec97ab691667052c1cf32cf05a874470d5683948ca1ca86bd9


Every field of the two records is equal except `provenance_iri`, the IRI of the test in its graph, which the record keeps for reporting only and no analysis step reads. ReL is empty in both, because neither graph gives a value for Zx3. Both records point to the same curve file on the data host (by construction: the added PMDco 3 curve tables link the files that PMDco 2 links, section 5), and the downloaded bytes are identical. From here on, the analysis cannot tell which generation the data came from; notebook 05 runs it on all ten specimens from both and compares the results.

## 5. The curve tables

A record is complete only with its raw curve: where the CSV file is, which column is time, crosshead travel, force and strain, and in which unit. Both generations say this with [CSVW](https://www.w3.org/TR/tabular-metadata/) tables, which each profile reads with its `CURVE_COLUMNS` query. The rows for Zx3:

In [9]:
print("pmdco2-tto, CURVE_COLUMNS:")
print(pmdco2_tto.CURVE_COLUMNS.query.strip())

curve_rows = {}
for profile_id, module, dataset in (
    ("pmdco2-tto", pmdco2_tto, "pmdco2_tto_example_parallel"),
    ("pmdco3-tto", pmdco3_tto, "pmdco3_tto_example_parallel"),
):
    query = module.CURVE_COLUMNS  # the profile's own query, with its own prefixes
    rows = sources.select({"kind": "ontodocker", "datasets": [dataset]}, query.query, list(query.columns))
    rows = rows[rows["process"] == records[profile_id].provenance_iri].drop(columns=["process", "source_part"])
    rows["unit"] = rows["unit"].map(short)
    curve_rows[profile_id] = rows.sort_values("column_number").set_index("role")
    print(f"{profile_id} ({dataset}):")
    display(curve_rows[profile_id])

pmdco2-tto, CURVE_COLUMNS:
PREFIX pmd:  <https://w3id.org/pmd/co/>
PREFIX csvw: <http://www.w3.org/ns/csvw#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?process ?url ?role ?column_number ?unit
WHERE {
  VALUES (?class ?role) {
    (pmd:Duration            "time")
    (pmd:CrossheadSeparation "crosshead")
    (pmd:Force               "force")
    (pmd:PercentageExtension "strain")
  }
  {
    # Position of each column in the csvw:column RDF list (1-based).
    SELECT ?process ?url ?column (COUNT(DISTINCT ?position) AS ?column_number)
    WHERE {
      ?process a pmd:TensileTest ;
               pmd:characteristic ?dataset .
      ?dataset a pmd:Dataset ;
               pmd:resource ?table .
      ?table a csvw:Table ;
             csvw:url ?url ;
             csvw:tableSchema ?schema .
      ?schema csvw:column/rdf:rest* ?position .
      ?position rdf:rest* ?cell .
      ?cell rdf:first ?column .
    }
    GROUP BY ?process ?url ?column
  }
  ?column a ?class .


pmdco2-tto (pmdco2_tto_example_parallel):


,url,column_number,unit
role,,,
time,https://data.pmds-mpcdf-2.pyiron.com/Zx3.csv,1,unit:SEC
crosshead,https://data.pmds-mpcdf-2.pyiron.com/Zx3.csv,2,unit:MilliM
force,https://data.pmds-mpcdf-2.pyiron.com/Zx3.csv,3,unit:KiloN
strain,https://data.pmds-mpcdf-2.pyiron.com/Zx3.csv,4,unit:PERCENT


pmdco3-tto (pmdco3_tto_example_parallel):


,url,column_number,unit,delimiter,header_row_count,decimal_char
role,,,,,,
time,https://data.pmds-mpcdf-2.pyiron.com/Zx3.csv,1,unit:SEC,;,2,","
crosshead,https://data.pmds-mpcdf-2.pyiron.com/Zx3.csv,2,unit:MilliM,;,2,","
force,https://data.pmds-mpcdf-2.pyiron.com/Zx3.csv,3,unit:KiloN,;,2,","
strain,https://data.pmds-mpcdf-2.pyiron.com/Zx3.csv,4,unit:PERCENT,;,2,","


The PMDco 2 graph declares no CSV dialect (no `csvw:Dialect`), nor does the original PMDco 3 graph. The curve tables that this demonstrator added to the PMDco 3 slices declare one (`;` as delimiter, two header rows, `,` as decimal mark), and acquisition checks the declaration against the layout its CSV parser reads (not shown here; `tests/test_pmdco3_curve_tables.py` covers a declaration the parser cannot read). The column names in the graphs, compared with the header rows of the file itself:

In [10]:
NAMES_PMDCO2 = """
SELECT ?role ?name ?title WHERE {
  VALUES (?class ?role) { (pmdco:Duration "time") (pmdco:CrossheadSeparation "crosshead") (pmdco:Force "force") (pmdco:PercentageExtension "strain") }
  ?table csvw:url ?url ; csvw:tableSchema/csvw:column/rdf:rest*/rdf:first ?column .
  FILTER(STR(?url) = "%s")
  ?column a ?class ; csvw:name ?name ; dc:title ?title .
}
"""
NAMES_PMDCO3 = """
SELECT ?role ?name ?title WHERE {
  VALUES (?class ?role) { (obo:BFO_0000202 "time") (tto:TTO_0000013 "crosshead") (pmdco:PMD_0020200 "force") (pmdco:PMD_0060011 "strain") }
  ?table a pmdco:PMD_0000992 ; csvw:url ?url ; csvw:tableSchema/csvw:column/rdf:rest*/rdf:first ?column .
  FILTER(STR(?url) = "%s")
  ?column obo:IAO_0000136/rdf:type ?class ; csvw:name ?name ; csvw:title ?title .
}
"""
url = records["pmdco3-tto"].curve_url
names = {
    "PMDco 2": select("pmdco2_tto_example_parallel", NAMES_PMDCO2 % url, ["role", "name", "title"]).set_index("role"),
    "PMDco 3": select("pmdco3_tto_example_parallel", NAMES_PMDCO3 % url, ["role", "name", "title"]).set_index("role"),
}
with urllib.request.urlopen(url, timeout=30) as response:  # token-free GET of the first bytes only
    header_lines = response.read(200).decode("utf-8").splitlines()[:2]
header_names, header_units = (line.split(";") for line in header_lines)

roles = list(curve_rows["pmdco3-tto"].index)
positions = [int(curve_rows["pmdco3-tto"].loc[role, "column_number"]) for role in roles]
display(pd.DataFrame({
    "column": positions,
    "PMDco 2 csvw:name": [names["PMDco 2"].loc[role, "name"] for role in roles],
    "PMDco 2 dc:title": [names["PMDco 2"].loc[role, "title"] for role in roles],
    "PMDco 3 csvw:title": [names["PMDco 3"].loc[role, "title"] for role in roles],
    "PMDco 3 csvw:name": [names["PMDco 3"].loc[role, "name"] for role in roles],
    "CSV header row 1": [header_names[position - 1] for position in positions],
    "CSV header row 2": [header_units[position - 1] for position in positions],
}, index=pd.Index(roles, name="role")))

,column,PMDco 2 csvw:name,PMDco 2 dc:title,PMDco 3 csvw:title,PMDco 3 csvw:name,CSV header row 1,CSV header row 2
role,,,,,,,
time,1,Time,Time,Time,Time,Time,(sec)
crosshead,2,Crosshead travel,Crosshead Separation,Crosshead separation,Crosshead_separation,Crosshead separation,(mm)
force,3,Force,Force,Load,Load,Load,(kN)
strain,4,Strain,Percentage Extension,Extensometer elongation,Extensometer_elongation,Extensometer elongation,(%)


In the added PMDco 3 curve tables, the titles repeat the CSV headers exactly; `csvw:name` is a sanitised identifier. The PMDco 2 names and titles differ from the headers. Both profiles locate each column by its position in the `csvw:column` list, not by name, and the CSV parser checks the unit row of the file against the unit the graph declares (not shown here; `tests/test_acquisition_curves.py` covers a mismatch).

### The PMDco 3 curve tables are additions

The original PMDco 3 graph in the TTO repository (commit `08a7aec`) links only one aggregate table, the lab's `S355_data.csv` with 57 columns of metadata and results for all ten tests, but none of the per-test curve files. This demonstrator added one curve table per test, with a modelling approved by the demonstrator's maintainer (the TTO maintainers have not reviewed it):

- the table is a `csvw:Table` and a `pmdco:PMD_0000992` (time series), linked from the test by `obo:OBI_0000299` (has specified output), with `csvw:url` pointing to the same CSV file that PMDco 2 links;
- each column is a `csvw:Column` and an `obo:IAO_0000109` (measurement datum) with its unit (`obo:IAO_0000039`), and is about (`obo:IAO_0000136`) a per-test entity typed with the quantity class: a temporal interval (`obo:BFO_0000202`) that the testing step occupies, the crosshead separation of the machine (`tto:TTO_0000013`), and the force (`pmdco:PMD_0020200`) and percentage extension (`pmdco:PMD_0060011`) of the test piece;
- the CSV layout is a named `csvw:Dialect` and a `csvw:NumericFormat`; the column order is an RDF list.

Each added table carries an `rdfs:comment` that says so, and the lab table carries one for its new URL on the data host. Readers of the graph can therefore tell the additions from the original:

In [11]:
MARKED = """
SELECT ?table ?url ?comment WHERE {
  ?table a csvw:Table ; csvw:url ?url ; rdfs:comment ?comment .
  FILTER(CONTAINS(STR(?comment), "PMD demonstrator"))
}
"""
marked = pd.concat(
    [select(dataset, MARKED, ["table", "url", "comment"]).assign(dataset=dataset) for dataset in GENERATIONS["pmdco3-tto"]],
    ignore_index=True,
)
marked["table"] = marked["table"].map(short)
display(marked.groupby("comment")["table"].agg(
    lambda tables: f"distinct tables: {len(set(tables))}, e.g. {sorted(set(tables))[0]}"
).to_frame())
print("Curve tables marked as added, per dataset:",
      marked[marked["comment"].str.startswith("Added")].groupby("dataset")["table"].nunique().to_dict())

,table
comment,
"Added by the PMD demonstrator (materialdigital/pmd-demonstrator, scripts/slice_pmdco3.py); not part of the TTO graph at commit 08a7aec.","distinct tables: 10, e.g. abox:Zd2_csvw_table"
"csvw:url relinked to the PMD data host by the PMD demonstrator (materialdigital/pmd-demonstrator, scripts/slice_pmdco3.py); file unchanged from the TTO repository at commit 08a7aec.","distinct tables: 1, e.g. abox:S355_data_csvw_table"


Curve tables marked as added, per dataset: {'pmdco3_tto_example_diagonal': 2, 'pmdco3_tto_example_parallel': 4, 'pmdco3_tto_example_perpendicular': 4}


## Summary

- PMDco 2 and PMDco 3 describe Zx3 with different structures: values on characteristics with readable class names, against qualities, measurement data and value specifications with numeric class IDs (section 2).
- Each profile keeps the ontology-specific SPARQL and unit aliases. Its conformance ASKs all hold on its own generation and none on the other (section 3).
- The exploration and acquisition nodes give the same specimens, orientations, records and lab values from both generations, and both point to the same curve bytes; only the provenance IRI differs (section 4).
- The PMDco 2 graph links the curve CSVs through CSVW tables. The original PMDco 3 graph links none; the curve tables that this demonstrator added to the PMDco 3 slices link the same files, declare the CSV dialect and repeat the exact headers (section 5).

The limits of these graphs and of the profiles' conventions are listed with those of the method in [notebook 02, section 11](02-tensile-method.ipynb).

Next: [notebook 04](04-tensile-workflow-knowledge-graph.ipynb) shows what the workflow itself records as a knowledge graph and joins it with these data graphs; [notebook 05](05-tensile-analysis-overview.ipynb) runs the analysis on all ten specimens from both generations; [notebook 06](06-tensile-results-publication.ipynb) publishes the results.